In [23]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [24]:
import pandas as pd
import numpy as np
df = pd.read_csv('fashion-mnist_train.csv')
df.head(3)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0


In [25]:
from sklearn.model_selection import train_test_split
X = df.drop('label', axis=1)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [26]:
X_train = X_train/255
X_test = X_test/255

In [27]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [28]:
import torch
X_train = torch.from_numpy(X_train).to(torch.float32)
X_test = torch.from_numpy(X_test).to(torch.float32)
y_train = torch.from_numpy(y_train).to(torch.long)
y_test = torch.from_numpy(y_test).to(torch.long)

In [29]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features.reshape(-1, 1, 28, 28)
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [30]:
train_ds = CustomDataset(X_train, y_train)
test_ds = CustomDataset(X_test, y_test)

In [31]:
import torch
import torch.nn as nn

class CNN(nn.Module):
  def __init__(self, num_conv_layers, num_filters, kernel_size, num_fc_layers, fc_layer_size, dropout_rate):
    super(CNN, self).__init__()
    layers = []
    in_channels = 1 #Grayscale images have only 1 input channel

    # Convolutional layers
    for i in range(num_conv_layers):
      layers.append(nn.Conv2d(in_channels, num_filters, kernel_size=kernel_size, padding='same'))
      layers.append(nn.ReLU())
      layers.append(nn.BatchNorm2d(num_filters))
      layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
      in_channels = num_filters # Update input channels for next layer

    self.features = nn.Sequential(*layers)

    # Fully connected layers
    fc_layers = [nn.Flatten()]
    input_size = num_filters*(28//(2**num_conv_layers))**2
    for i in range(num_fc_layers):
      fc_layers.append(nn.Linear(input_size, fc_layer_size))
      fc_layers.append(nn.ReLU())
      fc_layers.append(nn.Dropout(dropout_rate))
      input_size = fc_layer_size
    fc_layers.append(nn.Linear(input_size, 10))

    self.classifier = nn.Sequential(*fc_layers)

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [32]:
!pip install optuna

In [33]:
import optuna

# Objective function
def objective(trial):
  # Hyperparameter values
  num_conv_layers = trial.suggest_int('num_conv_layers', 1, 3)
  num_filters = trial.suggest_categorical('num_filters', [16, 32, 64])
  kernel_size = trial.suggest_categorical('kernel_size', [3, 5])

  num_fc_layers = trial.suggest_int('num_fc_layers', 1, 3)
  fc_layer_size = trial.suggest_categorical('fc_layer_size', [64, 128])

  dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.5, step=0.1)
  weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
  learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
  optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])
  batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
  epochs = trial.suggest_int('epochs', 10, 30, step=10)

  # Model
  model = CNN(num_conv_layers, num_filters, kernel_size, num_fc_layers, fc_layer_size, dropout_rate)
  model.to(device)

  # Data
  train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
  test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True)

  # Loss function
  loss_function = nn.CrossEntropyLoss()

  # Optimizer selection
  if optimizer_name == 'Adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # training loop
  for epoch in range(epochs):
    for batch_features, batch_labels in train_loader:
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)
      y_pred = model(batch_features)
      loss = loss_function(y_pred, batch_labels)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

    # evaluation
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
      for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        y_pred = model(batch_features)
        _, predicted = torch.max(y_pred, 1)
        total += batch_labels.shape[0]
        correct += (predicted == batch_labels).sum().item()

    accuracy = correct/total
    trial.report(accuracy, epoch)
    if trial.should_prune():
      raise optuna.exceptions.TrialPruned()

  return accuracy

In [34]:
import optuna
study = optuna.create_study(direction='maximize')

[I 2026-01-09 09:26:56,126] A new study created in memory with name: no-name-f8d174c8-a2ad-49b4-a05c-d80475adbb0d


In [35]:
study.optimize(objective, n_trials=5)

[I 2026-01-09 09:29:16,830] Trial 0 finished with value: 0.91325 and parameters: {'num_conv_layers': 3, 'num_filters': 32, 'kernel_size': 3, 'num_fc_layers': 3, 'fc_layer_size': 128, 'dropout_rate': 0.4, 'weight_decay': 5.418261389718129e-05, 'learning_rate': 0.0025141119886116503, 'optimizer': 'RMSprop', 'batch_size': 32, 'epochs': 20}. Best is trial 0 with value: 0.91325.
[I 2026-01-09 09:32:37,899] Trial 1 finished with value: 0.9234166666666667 and parameters: {'num_conv_layers': 2, 'num_filters': 64, 'kernel_size': 3, 'num_fc_layers': 2, 'fc_layer_size': 128, 'dropout_rate': 0.30000000000000004, 'weight_decay': 0.00025530990703157767, 'learning_rate': 0.00028123974477118995, 'optimizer': 'Adam', 'batch_size': 16, 'epochs': 20}. Best is trial 1 with value: 0.9234166666666667.
[I 2026-01-09 09:34:29,385] Trial 2 finished with value: 0.9089166666666667 and parameters: {'num_conv_layers': 3, 'num_filters': 64, 'kernel_size': 3, 'num_fc_layers': 2, 'fc_layer_size': 128, 'dropout_rate':

In [36]:
study.best_value

0.9234166666666667

In [37]:
study.best_params

{'num_conv_layers': 2,
 'num_filters': 64,
 'kernel_size': 3,
 'num_fc_layers': 2,
 'fc_layer_size': 128,
 'dropout_rate': 0.30000000000000004,
 'weight_decay': 0.00025530990703157767,
 'learning_rate': 0.00028123974477118995,
 'optimizer': 'Adam',
 'batch_size': 16,
 'epochs': 20}